In [1]:
!pip install roboflow ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.6/333.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 132.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
from roboflow import Roboflow
import os, shutil

rf = Roboflow(api_key="609NvYiApjVQrp8SeA6H")

project = rf.workspace("kunsh-kaul").project("unified_microplasticdetect_dataset")
dataset_version = project.version(2)

# Download locally (fast, reliable)
LOCAL_DOWNLOAD = "/content/yolov26_dataset"
dataset = dataset_version.download("yolov8", location=LOCAL_DOWNLOAD)

print("Downloaded locally to:", dataset.location)
print(os.listdir(dataset.location))

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/yolov26_dataset in yolov8:: 100%|██████████| 12891/12891 [00:02<00:00, 5928.29it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Downloaded locally to: /content/yolov26_dataset
['valid', 'train', 'test', 'data.yaml', 'README.dataset.txt', 'README.roboflow.txt']


In [12]:
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(LOCAL_DOWNLOAD, split, "images")
    if os.path.exists(img_dir):
        print(split, "images:", len(os.listdir(img_dir)))
    else:
        print(split, "folder missing")

train images: 5678
valid images: 490
test images: 275


In [13]:
DRIVE_TARGET = "/content/drive/MyDrive/IPD/Yolov26"

if os.path.exists(DRIVE_TARGET):
    shutil.rmtree(DRIVE_TARGET)

shutil.copytree(LOCAL_DOWNLOAD, DRIVE_TARGET)
print("Copied to Drive:", DRIVE_TARGET)
print(os.listdir(DRIVE_TARGET))

Copied to Drive: /content/drive/MyDrive/IPD/Yolov26
['valid', 'train', 'test', 'data.yaml', 'README.dataset.txt', 'README.roboflow.txt']


In [14]:
import yaml, os

DATASET_PATH = "/content/drive/MyDrive/IPD/Yolov26"

fixed_yaml = {
    "train": os.path.join(DATASET_PATH, "train", "images"),
    "val": os.path.join(DATASET_PATH, "valid", "images"),
    "test": os.path.join(DATASET_PATH, "test", "images"),
    "nc": 3,
    "names": ["Beams", "Foams", "Microplastic"]
}

DATA_YAML = "/content/data_yolov26.yaml"
with open(DATA_YAML, "w") as f:
    yaml.dump(fixed_yaml, f)

print(open(DATA_YAML).read())

names:
- Beams
- Foams
- Microplastic
nc: 3
test: /content/drive/MyDrive/IPD/Yolov26/test/images
train: /content/drive/MyDrive/IPD/Yolov26/train/images
val: /content/drive/MyDrive/IPD/Yolov26/valid/images



In [15]:
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_PATH, split, "images")
    lbl_dir = os.path.join(DATASET_PATH, split, "labels")
    print(split, "images:", len(os.listdir(img_dir)), "labels:", len(os.listdir(lbl_dir)))

train images: 5678 labels: 5678
valid images: 490 labels: 490
test images: 275 labels: 275


In [16]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    cache=False,   # cache=True would try to load all images into RAM, risky with Drive I/O; keep False
    project="/content/runs_v26",
    name="microplastic_yolov26"
)

Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_yolov26.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=microplastic_yolov26, nbs=64, nms=None, opset=None, optimize=Fa

In [17]:
metrics = model.val(data=DATA_YAML, plots=True)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,375,421 parameters, 0 gradients, 5.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 28.4±9.1 MB/s, size: 32.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/.shortcut-targets-by-id/1HoU5E2t_LoQXbHeH7ZSAszBAwlKCf7Nf/IPD/Yolov26/valid/labels.cache... 490 images, 50 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 490/490 128.5Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 1027, len(boxes) = 4065. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 31/31 3.6it/s 

In [19]:
import shutil, os

dest_dir = "/content/drive/MyDrive/IPD/runs/microplastic_yolov26"
os.makedirs(os.path.dirname(dest_dir), exist_ok=True)
shutil.copytree("/content/runs_v26/microplastic_yolov26", dest_dir, dirs_exist_ok=True)
print("Copied to:", dest_dir)

Copied to: /content/drive/MyDrive/IPD/runs/microplastic_yolov26
